# Statistical testing

Checking two simple properties, clip duration and loudness (RMS energy),
for real class differences before any modeling.

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy import stats

df = pd.read_pickle('data/audio_metadata.pkl')

rms_values = []
for fname in df.slice_file_name:
    y, sr = librosa.load(os.path.join('data', fname), sr = 22050)
    rms_values.append(np.sqrt(np.mean(y ** 2)))
df['rms'] = rms_values
df.shape

(450, 7)

In [2]:
groups_duration = [g.duration.values for _, g in df.groupby('class')]
h_dur, p_dur = stats.kruskal(*groups_duration)
print(f'Kruskal-Wallis, duration by class: H = {h_dur:.1f}, p = {p_dur:.2e}')

Kruskal-Wallis, duration by class: H = 241.3, p = 6.85e-47


In [3]:
groups_rms = [g.rms.values for _, g in df.groupby('class')]
h_rms, p_rms = stats.kruskal(*groups_rms)
print(f'Kruskal-Wallis, RMS energy by class: H = {h_rms:.1f}, p = {p_rms:.2e}')

df.groupby('class').rms.mean().sort_values()

Kruskal-Wallis, RMS energy by class: H = 87.1, p = 6.12e-15


class
children_playing    0.034922
air_conditioner     0.061543
siren               0.065592
jackhammer          0.066939
dog_bark            0.069786
street_music        0.072302
engine_idling       0.074075
drilling            0.093126
car_horn            0.096121
gun_shot            0.162830
Name: rms, dtype: float32

Both are real, significant differences. Duration varies by class since
UrbanSound8K clips are trimmed around actual sound events, a gunshot
event is much shorter than a sustained engine idle. RMS energy (a
simple loudness measure) also differs meaningfully: `gun_shot` averages
about 4.7x louder than `children_playing`, a sharp loud transient
versus ambient background noise, exactly matching what these labels
describe. Neither duration nor RMS alone is a real classifier
(clips of very different sounds can share similar duration or
loudness), but both are legitimate, cheap-to-compute signals worth
checking for before reaching for spectral features.

In [4]:
import json, os
os.makedirs('outputs', exist_ok = True)
with open('outputs/statistical_tests.json', 'w') as f:
    json.dump({
        'duration_kruskal_h': float(h_dur),
        'duration_kruskal_p': float(p_dur),
        'rms_kruskal_h': float(h_rms),
        'rms_kruskal_p': float(p_rms),
    }, f, indent = 2)